# 🧬 Workshop Demo: From Raw Biological Data to Usable Dataset
### A Realistic Simulation of a Bioinformatics Preprocessing Pipeline

**Requirements:** Python 3 + pandas only. No specialized libraries needed.

**What we'll do:**
1. Simulate raw sequencing reads (FASTQ-like)
2. Calculate QC metrics on each read
3. Inspect the data quality before filtering
4. Apply quality filters to remove bad reads
5. Transform clean sequences into a structured feature table
6. Produce a per-sample summary ready for downstream analysis

---

In [13]:
# @title
# CELL 1: Setup and Imports

import pandas as pd
import random

pd.set_option('display.max_colwidth', 65)
pd.set_option('display.max_rows', 20)

print('Libraries loaded.')

Libraries loaded.


---
## Step 1: Simulate Raw Sequencing Reads

In reality, a sequencer produces a `.fastq` file with millions of reads.  
Each read has:
- A **header** (read ID + metadata)
- A **sequence** (string of A/T/G/C/N)
- A **quality string** (ASCII-encoded Phred scores per base)

We'll simulate 30 reads with a realistic mix of quality levels.

In [14]:
# CELL 2: Simulate Raw FASTQ-like Data
random.seed(42)

bases = ['A', 'T', 'G', 'C']
bad_bases = ['A', 'T', 'G', 'C', 'N']  # N = unknown base (machine couldn't read)

def generate_read(length=60, bad=False):
    """Simulate a sequencing read. Bad reads contain N (unknown) bases."""
    pool = bad_bases if bad else bases
    return ''.join(random.choices(pool, k=length))

def generate_quality(length=60, low_quality=False):
    """
    Simulate Phred quality scores encoded as ASCII characters.
    Phred+33 encoding: '!' = score 0 (worst), 'I' = score 40 (best)
    Low-quality reads have '!!!' at the end (machine lost confidence).
    """
    if low_quality:
        good_part = ''.join(random.choices('HGFEDCBA@?', k=length // 2))
        bad_part = '!' * (length - length // 2)  # Bad bases at end
        return good_part + bad_part
    else:
        return ''.join(random.choices('IIIHHHGGF', k=length))

# Generate 30 reads: good, low-quality, short, and N-heavy
reads_data = []

for i in range(1, 31):
    if i <= 15:          # Good reads — should pass all filters
        seq = generate_read(60, bad=False)
        qual = generate_quality(60, low_quality=False)
        sample = 'Sample_A'
        read_type = 'good'
    elif i <= 22:        # Low quality reads — should be filtered out
        seq = generate_read(60, bad=False)
        qual = generate_quality(60, low_quality=True)
        sample = 'Sample_A'
        read_type = 'low_quality'
    elif i <= 26:        # Short reads — should be filtered out
        length = random.randint(20, 35)
        seq = generate_read(length, bad=False)
        qual = generate_quality(length, low_quality=False)
        sample = 'Sample_B'
        read_type = 'short'
    else:                # N-heavy reads — should be filtered out
        seq = generate_read(60, bad=True)
        qual = generate_quality(60, low_quality=False)
        sample = 'Sample_B'
        read_type = 'n_heavy'

    reads_data.append({
        'read_id': f'Read_{i:03d}',
        'sample': sample,
        'sequence': seq,
        'quality': qual,
        'true_type': read_type  # We know this only because we simulated it
    })

raw_df = pd.DataFrame(reads_data)
print(f' Raw dataset: {len(raw_df)} reads loaded')
print(f'Samples: {list(raw_df["sample"].unique())}')
print(f'Read types (ground truth, known only because we simulated):')
print(raw_df['true_type'].value_counts())
print()
raw_df[['read_id', 'sample', 'sequence', 'quality']].head(8)

 Raw dataset: 30 reads loaded
Samples: ['Sample_A', 'Sample_B']
Read types (ground truth, known only because we simulated):
true_type
good           15
low_quality     7
short           4
n_heavy         4
Name: count, dtype: int64



,read_id,sample,sequence,quality
0,Read_001,Sample_A,GATAGGCATAAGAAGGAGCACGTACTAACGCGGCTGCGCGGAATAAATGTTATCGGAGAT,FHHGGGIIIIIFGIHHFHIIHIHFHIFHIIIHGHIHFHFGIGGHIHIHHFGIHIFGIHHI
1,Read_002,Sample_A,CGCGATACCCTACCATACCATGTCTAGGATCGTGAATGAAAGACCAAGAACGTCCAATTT,GGFIHHGIIHHIIFHGHIFGFFGIHIHIHFIGHHFFHGIIFHHGIHHGIFIIHGIIFIHH
2,Read_003,Sample_A,TGGCAGATGTTCATCCAATCCCTACGGCGACTGCAAAGTGGAGTTCCATTACGTGGTAAC,FHGHIIIFGGFIIIGGHHIFGFGGIGIFGGGIGIGGIGHIGIIIIGFIHHFHFIFIFIIH
3,Read_004,Sample_A,GTGGTGTGACGGGGTAGTTCGTTTTTATCGCGTGATTGGTTATCCAAGGTCCGAAAATCA,HHIGHIHIGGIHIFHIIGHGGFHFFHGHGHFGHIIHGHHIIIIIHIIGGIHHHIHFHGGG
4,Read_005,Sample_A,TATCCCTGGGAATATGTATAGATGAGATATATCTCCTAAGCTGCGCCAAAAGGAGCAGGA,GFIIIGFHGIGHIGGHIFGHHHHHGGIFHGGHGFIGHHHGIGGIGIHHHIGIIGHGGIIF
5,Read_006,Sample_A,AGTCGGTCAAGTGTGTGTGTTAAAGCAATGCGTAAAAAGTCGGACTCGTTAACTCTAGAA,HIFIGHGIHHHGFIHHGFIIHIIIIHHIIGGHGFGHHFHHHFGIIIGHIIGGFHHIIHII
6,Read_007,Sample_A,ACCGCCGTACTGCAGGTATTGCTGTCCGTTTCTGCGGTATCGCAGCTGACGAAGGACCTA,GHGHIGFIHHFHGGIGHFHGIIHFHIHHHHHIIGHIGIGGGHHGFHIHHGGHHHGHHIHF
7,Read_008,Sample_A,TGGCCCTTGCCAAGCCGTAGCTGCACTTAGGCAACGACATTTACCGATTGGTTCATTAGG,FIFHIHGGIHHFIGHHHHGGHGHFHIHHGIIGIIHFHFGIGHGGHIFIFGGFFGHGIHHG


---
## Step 2: Quality Control — Calculate QC Metrics

We calculate the same metrics that **FastQC** (the industry-standard QC tool) computes:
- **Average Phred quality score** — how confident is the machine overall?
- **Read length** — is the read long enough to be useful?
- **N fraction** — what proportion of bases are unknown?
- **GC content** — is the base composition biologically plausible?

In [15]:
# CELL 3: Calculate QC Metrics

def calc_avg_quality(quality_string):
    """
    Convert ASCII Phred+33 quality string to average Phred score.
    Formula: ASCII value - 33 = Phred score
    Phred 30 = 99.9% accuracy | Phred 20 = 99% accuracy | Phred 10 = 90%
    """
    scores = [ord(c) - 33 for c in quality_string]
    return round(sum(scores) / len(scores), 2)

def calc_n_fraction(sequence):
    """Fraction of bases that are unknown (N). Higher = worse."""
    return round(sequence.count('N') / len(sequence), 4)

def calc_gc_content(sequence):
    """GC content = fraction of G + C bases (should be ~0.4–0.6 for most organisms)."""
    clean = [b for b in sequence if b in 'ATGC']
    if not clean:
        return 0.0
    return round((sequence.count('G') + sequence.count('C')) / len(clean), 4)

# Apply QC calculations to all reads
raw_df['avg_quality'] = raw_df['quality'].apply(calc_avg_quality)
raw_df['read_length'] = raw_df['sequence'].apply(len)
raw_df['n_fraction'] = raw_df['sequence'].apply(calc_n_fraction)
raw_df['gc_content'] = raw_df['sequence'].apply(calc_gc_content)

print(' QC Metrics Calculated for All Reads')
print('\nQuality Score Summary:')
print(raw_df['avg_quality'].describe().round(2))
print(f'\nReads with any N bases: {(raw_df["n_fraction"] > 0).sum()} / {len(raw_df)}')
raw_df[['read_id', 'sample', 'avg_quality', 'read_length', 'n_fraction', 'gc_content']].head(10)

 QC Metrics Calculated for All Reads

Quality Score Summary:
count    30.00
mean     33.87
std       9.24
min      16.97
25%      38.62
50%      38.83
75%      38.96
max      39.07
Name: avg_quality, dtype: float64

Reads with any N bases: 4 / 30


,read_id,sample,avg_quality,read_length,n_fraction,gc_content
0,Read_001,Sample_A,38.88,60,0.0,0.4833
1,Read_002,Sample_A,38.77,60,0.0,0.4500
2,Read_003,Sample_A,38.70,60,0.0,0.5000
3,Read_004,Sample_A,38.85,60,0.0,0.4667
4,Read_005,Sample_A,38.72,60,0.0,0.4333
5,Read_006,Sample_A,38.92,60,0.0,0.4167
6,Read_007,Sample_A,38.73,60,0.0,0.5667
7,Read_008,Sample_A,38.60,60,0.0,0.5167
8,Read_009,Sample_A,39.03,60,0.0,0.4667
9,Read_010,Sample_A,39.07,60,0.0,0.5167


---
## Step 3: QC Report — Inspect Data Quality Before Filtering

Before we decide what to remove, we inspect the QC metrics.  
This is equivalent to reading a **FastQC HTML report**.

In [16]:
# CELL 4: QC Report — Pre-Filtering Inspection

QUALITY_THRESHOLD = 20   # Phred >= 20 required (99% accuracy per base)
LENGTH_THRESHOLD = 40    # Read must be >= 40 bases long
N_THRESHOLD = 0.05       # Maximum 5% unknown bases allowed

total = len(raw_df)
would_fail_quality = (raw_df['avg_quality'] < QUALITY_THRESHOLD).sum()
would_fail_length = (raw_df['read_length'] < LENGTH_THRESHOLD).sum()
would_fail_n = (raw_df['n_fraction'] > N_THRESHOLD).sum()

print('=' * 5)
print('           QC REPORT — BEFORE FILTERING')
print('=' * 5)
print(f'  Total reads in dataset:                      {total}')
print(f'  Average quality score:                       {raw_df["avg_quality"].mean():.2f} (Phred)')
print(f'  Average read length:                         {raw_df["read_length"].mean():.1f} bp')
print(f'  Reads below quality threshold (Phred < {QUALITY_THRESHOLD}): {would_fail_quality} ({would_fail_quality/total*100:.1f}%)')
print(f'  Reads below length threshold (< {LENGTH_THRESHOLD} bp):      {would_fail_length} ({would_fail_length/total*100:.1f}%)')
print(f'  Reads with too many N bases (> {N_THRESHOLD*100:.0f}%):         {would_fail_n} ({would_fail_n/total*100:.1f}%)')
print('_' * 65)
print(f'  These reads must be removed before analysis.')

print('\n--- Quality by Sample ---')
print(raw_df.groupby('sample')[['avg_quality', 'read_length', 'n_fraction']]
      .mean().round(3))

=====
           QC REPORT — BEFORE FILTERING
=====
  Total reads in dataset:                      30
  Average quality score:                       33.87 (Phred)
  Average read length:                         56.0 bp
  Reads below quality threshold (Phred < 20): 7 (23.3%)
  Reads below length threshold (< 40 bp):      4 (13.3%)
  Reads with too many N bases (> 5%):         4 (13.3%)
_________________________________________________________________
  These reads must be removed before analysis.

--- Quality by Sample ---
          avg_quality  read_length  n_fraction
sample                                        
Sample_A       32.033       60.000       0.000
Sample_B       38.921       45.125       0.125


---
## Step 4: Filtering — Remove Reads That Fail Quality Thresholds

We apply three filters:
1. **Quality filter:** Phred score ≥ 20 → at least 99% accuracy per base
2. **Length filter:** Read ≥ 40 bp → long enough to map reliably
3. **N filter:** ≤ 5% unknown bases → too many unknowns = unreliable

Each rejected read gets a **reason code** — good practice for reproducibility.

In [17]:
# CELL 5: Apply Quality Filters

def passes_qc(row):
    """Return (passed: bool, reason: str) for each read."""
    if row['avg_quality'] < QUALITY_THRESHOLD:
        return False, 'low_quality'
    if row['read_length'] < LENGTH_THRESHOLD:
        return False, 'too_short'
    if row['n_fraction'] > N_THRESHOLD:
        return False, 'too_many_N'
    return True, 'passed'

results = raw_df.apply(passes_qc, axis=1)
raw_df['passed_qc'] = [r[0] for r in results]
raw_df['fail_reason'] = [r[1] for r in results]

clean_df = raw_df[raw_df['passed_qc'] == True].copy()
removed_df = raw_df[raw_df['passed_qc'] == False].copy()

print(' FILTERING RESULTS')
print(f'  Before filtering:  {len(raw_df):>4} reads')
print(f'  After filtering:   {len(clean_df):>4} reads')
print(f'  Removed:           {len(removed_df):>4} reads ({len(removed_df)/len(raw_df)*100:.1f}%)')
print()
print('Removed reads — reason breakdown:')
print(removed_df['fail_reason'].value_counts())

 FILTERING RESULTS
  Before filtering:    30 reads
  After filtering:     15 reads
  Removed:             15 reads (50.0%)

Removed reads — reason breakdown:
fail_reason
low_quality    7
too_short      4
too_many_N     4
Name: count, dtype: int64


---
## Step 5: Transformation — Sequences → Numeric Feature Table

Text strings can't be used directly in statistics or machine learning.  
We **transform** each clean sequence into numeric features.

Here we extract base composition features.  
In real pipelines: this step would be alignment → gene count matrix.

In [18]:
# CELL 6: Transform Sequences → Numeric Feature Table

def extract_features(row):
    """
    Extract numeric features from a clean sequence.

    In real bioinformatics:
    - RNA-seq: reads are aligned to a genome → count how many land on each gene
    - Result: count matrix (rows=samples, columns=genes)

    Here: we compute base composition as our features (simpler but same concept).
    """
    seq = row['sequence']
    length = len(seq)
    return {
        'read_id': row['read_id'],
        'sample': row['sample'],
        'length': length,
        'pct_A': round(seq.count('A') / length * 100, 2),
        'pct_T': round(seq.count('T') / length * 100, 2),
        'pct_G': round(seq.count('G') / length * 100, 2),
        'pct_C': round(seq.count('C') / length * 100, 2),
        'gc_content': round((seq.count('G') + seq.count('C')) / length * 100, 2),
        'at_content': round((seq.count('A') + seq.count('T')) / length * 100, 2),
        'avg_quality': row['avg_quality']
    }

feature_rows = clean_df.apply(extract_features, axis=1, result_type='expand')
features_df = feature_rows.copy()
print(f'  Structured feature matrix: {features_df.shape[0]} rows × {features_df.shape[1]} columns')
print()
print('First 5 rows of our structured dataset:')
features_df.head()

  Structured feature matrix: 15 rows × 10 columns

First 5 rows of our structured dataset:


,read_id,sample,length,pct_A,pct_T,pct_G,pct_C,gc_content,at_content,avg_quality
0,Read_001,Sample_A,60,33.33,18.33,31.67,16.67,48.33,51.67,38.88
1,Read_002,Sample_A,60,33.33,21.67,18.33,26.67,45.00,55.00,38.77
2,Read_003,Sample_A,60,25.00,25.00,25.00,25.00,50.00,50.00,38.70
3,Read_004,Sample_A,60,20.00,33.33,31.67,15.00,46.67,53.33,38.85
4,Read_005,Sample_A,60,33.33,23.33,25.00,18.33,43.33,56.67,38.72
